# Fine-tuning a local LLM to write the best coaching prompt for a coding agent

A local LoRA-tuned G_theta proposes a short suggestion y (signature + plan +
edge cases); a frozen remote black-box O (an OpenAI model) writes code from y
alone; Eval_x(z) = f in [0,1] scores it against hidden unit tests. O is never
differentiated -- only log p_theta(y|x) is.

Four estimators are compared at a matched O-call budget: `reinforce_b`,
`ours_test`, `qcv`, `qcv_delta`. See `black-box DFL/mcp_prototype_plan.md` for
the framework and `toy_token_q/` for the rigorous multi-seed version.

## The estimator, in one screen

The score-function gradient is unbiased for *any* per-position coefficient a_t:

$$\hat g=\sum_t a_t\,\nabla_\theta\log p_\theta(y_t\mid y_{<t},x)$$

- **`reinforce_b`**: a_t = f - b, one EMA baseline per problem.
- **`ours_test`** (draft Sec 3.2): split the credit across free local checks
  u_m plus the one real O-call, each with its own baseline b_m:
  $$a_t=\sum_m \mathbb{1}\{t\in S_m\}/|S_m|\;(d_m-b_m)$$
  Every S_m is the full generated span here (no per-token slots).

`qcv`/`qcv_delta` go beyond this form -- they add a term that is not a
reweighting of log p_theta -- and appear further down.

In [1]:
import gc
import os, json, re

# Must run BEFORE torch initializes CUDA; the first id listed is the one used.
os.environ["CUDA_VISIBLE_DEVICES"] = "4,5,6,3"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import torch
import torch.nn.functional as tF
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from openai import OpenAI
import matplotlib.pyplot as plt
from dotenv import load_dotenv

load_dotenv()   # HF_TOKEN / OPENAI_API_KEY from a local .env

True

In [2]:
from dotenv import load_dotenv
import os

print("cwd:", os.getcwd())
print(".env found and loaded:", load_dotenv())
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

cwd: /nethome/zzhao628/blogs/black-box DFL/mcp_dfl_prototype
.env found and loaded: True
HF_TOKEN set: True
OPENAI_API_KEY set: True


In [3]:
# MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
O_MODEL = "gpt-5-nano"          # the frozen O
T_MAX = 96                      # max generated tokens for y

device = "cuda" if torch.cuda.is_available() else "cpu"
hf_token = os.environ.get("HF_TOKEN")   # gated models only

tok = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16,
                                            token=hf_token).to(device)
for p in base.parameters():
    p.requires_grad_(False)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [4]:
lora_cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
                      lora_dropout=0.0, task_type="CAUSAL_LM")
model = get_peft_model(base, lora_cfg)
model.print_trainable_parameters()

_init_lora_state = {k: v.clone() for k, v in model.state_dict().items() if "lora" in k}

def reset_lora():
    sd = model.state_dict()
    sd.update(_init_lora_state)
    model.load_state_dict(sd)

api_key = os.environ.get("OPENAI_API_KEY")
client = OpenAI(api_key=api_key) if api_key else None

trainable params: 2,293,760 || all params: 3,215,043,584 || trainable%: 0.0713


## A toy coding world

Small LeetCode-style problems with edge cases a cheap O gets wrong often
enough to give `Eval_x(z)` real variance. (Tool-routing was too easy: `f` was
0.00 every step, so y's quality never mattered.) Hidden `tests` are never
shown to G_theta or O.

In [5]:
PROBLEMS = [
    {"id": "second_largest",
     "spec": ("Write a Python function `solve(nums)` that returns the second-largest "
              "UNIQUE value in the list `nums`. If `nums` has fewer than 2 distinct "
              "values, return None."),
     "tests": [(([3, 1, 2],), 2), (([5, 5, 5],), None), (([10],), None),
               (([-1, -2, -3, -2],), -2), (([4, 4, 1, 2, 2],), 2)]},
    {"id": "is_balanced",
     "spec": ("Write a Python function `solve(s)` that returns True if every bracket "
              "in the string `s` (any of `()[]{}`; other characters are ignored) is "
              "closed by the matching type in the correct order, else False."),
     "tests": [(("([]){}",), True), (("([)]",), False), (("",), True),
               (("(a[b]{c}d)",), True), (("(",), False)]},
    {"id": "run_length_encode",
     "spec": ("Write a Python function `solve(s)` that run-length-encodes the string "
              "`s`, e.g. 'aaabbc' -> 'a3b2c1'. Return '' for an empty string."),
     "tests": [(("aaabbc",), "a3b2c1"), (("",), ""), (("a",), "a1"),
               (("aabbaa",), "a2b2a2")]},
    {"id": "longest_unique_run",
     "spec": ("Write a Python function `solve(s)` that returns the length of the "
              "longest substring of `s` with no repeated characters."),
     "tests": [(("abcabcbb",), 3), (("bbbbb",), 1), (("pwwkew",), 3),
               (("",), 0), (("dvdf",), 3)]},
    {"id": "merge_intervals",
     "spec": ("Write a Python function `solve(intervals)` that merges all overlapping "
              "[start, end] pairs in the list `intervals` and returns the merged "
              "list sorted by start."),
     "tests": [(([[1, 3], [2, 6], [8, 10], [15, 18]],), [[1, 6], [8, 10], [15, 18]]),
               (([[1, 4], [4, 5]],), [[1, 5]]),
               (([],), [])]},
    {"id": "majority_element",
     "spec": ("Write a Python function `solve(nums)` that returns the value "
              "appearing more than len(nums)//2 times in `nums`. Assume one "
              "always exists."),
     "tests": [(([3, 2, 3],), 3), (([2, 2, 1, 1, 1, 2, 2],), 2), (([1],), 1)]},
]

## G_theta: one rollout

Two-pass RLHF-style rollout: sample without grad, then one teacher-forced
forward with grad to read off log p_theta.

In [6]:
from contextlib import contextmanager, nullcontext
from typing import Any

# Left-pad so the generated span starts at the same column for every row.
tok.padding_side = "left"

try:                                    # torch >= 2.2
    from torch.nn.attention import sdpa_kernel, SDPBackend

    def _math_sdpa():
        return sdpa_kernel(SDPBackend.MATH)
except Exception:
    def _math_sdpa():
        return nullcontext()


@contextmanager
def double_backward_attn():
    """Attention path that survives create_graph=True: the fused SDPA backward
    is not itself differentiable. Only qcv_delta needs it."""
    prev = getattr(base.config, "_attn_implementation", None)
    if prev is not None:
        base.config._attn_implementation = "eager"
    try:
        with _math_sdpa():
            yield
    finally:
        if prev is not None:
            base.config._attn_implementation = prev


def build_prompt(problem_spec):
    system = (
        "You are a coding coach. You will be shown a problem statement, but the "
        "engineer who implements it will NOT see this statement -- they will see "
        "only your guidance. Restate the exact function signature, give a short "
        "approach, and list the edge cases that matter. Respond with exactly one "
        'line of JSON: {"signature": "<fn_name(args)>", "plan": "<1-2 sentence '
        'approach>", "edge_cases": ["<case>", ...]}.'
    )
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": problem_spec}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def rollout_batch(problem_specs, double_backward=False, gens=None, need_h=True):
    """Sample B sequences at once and read off log p_theta for each.

    gens=... re-scores already sampled sequences (deterministic, no new O-calls);
    need_h=False skips output_hidden_states (~3.7GB at B=8).
    """
    prompts = [build_prompt(s) for s in problem_specs]
    enc = tok(prompts, return_tensors="pt", padding=True,
              add_special_tokens=False).to(device)
    PL = enc.input_ids.shape[1]

    if gens is None:
        with torch.no_grad():           # sampling keeps the fast fused kernels
            gens = model.generate(**enc, do_sample=True, temperature=1.0,
                                  max_new_tokens=T_MAX,
                                  pad_token_id=tok.pad_token_id)
    gen_part = gens[:, PL:]                                    # (B, Tg)
    Tg = gen_part.shape[1]

    # Per-row length: up to and including the first stop token.
    is_pad = gen_part.eq(tok.pad_token_id)
    lens = torch.where(is_pad.any(1),
                       is_pad.to(torch.uint8).argmax(1) + 1,
                       torch.full((gen_part.shape[0],), Tg, device=gens.device))
    lens = lens.clamp(max=Tg)
    valid = torch.arange(Tg, device=gens.device)[None] < lens[:, None]
    attn = torch.cat([enc.attention_mask, valid.long()], dim=1)

    # RoPE positions from the mask, not a bare arange: with left padding row i's
    # real tokens do not start at 0, and scoring would disagree with sampling.
    pos = (attn.long().cumsum(-1) - 1).clamp(min=0)
    pos = pos.masked_fill(attn == 0, 0)

    with (double_backward_attn() if double_backward else nullcontext()):
        out = model(input_ids=gens, attention_mask=attn, position_ids=pos,
                    output_hidden_states=need_h)

    span = slice(PL - 1, PL - 1 + Tg)   # logits at PL-1+t predict token t
    logits_b = out.logits[:, span, :]                          # (B, Tg, V) bf16
    hid_b = out.hidden_states[-1][:, span, :].detach() if need_h else None

    items = []
    for i in range(len(problem_specs)):
        Ti = int(lens[i])
        y_i = gen_part[i, :Ti]
        # Cast per row: a full (B, Tg, V) float copy is ~800MB at B=16.
        lg = logits_b[i, :Ti, :].float()                        # (Ti, V)
        lp = tF.log_softmax(lg, dim=-1)[torch.arange(Ti, device=lg.device), y_i]
        items.append(dict[str, Any](y_text=tok.decode(y_i, skip_special_tokens=True),
                          lp=lp, y_ids=y_i,
                          # h_t predicts position t, so it sees y_{<t} only.
                          h=hid_b[i, :Ti, :] if need_h else None, logits=lg))
    return items, gens


def rollout(problem_spec, double_backward=False, gen=None):
    """Single-sample wrapper over rollout_batch."""
    items, gens = rollout_batch([problem_spec], double_backward, gen)
    it = items[0]
    return it["y_text"], it["lp"], it["y_ids"], it["h"], it["logits"], gens

In [7]:
# Sanity check (no O-call): gradient must reach LoRA only, never the frozen base.
y_text, lp, y_ids, h, logits, gen = rollout(PROBLEMS[0]["spec"])
print("sample y:", y_text)
print("h:", tuple(h.shape), " logits:", tuple(logits.shape))
lp.sum().backward()
lora_touched = any(p.grad is not None and p.grad.abs().sum() > 0
                   for n, p in model.named_parameters() if "lora" in n)
base_touched = any(p.grad is not None
                   for n, p in model.named_parameters() if p.requires_grad is False)
print("LoRA params received gradient:", lora_touched, "(should be True)")
print("Frozen base params received gradient:", base_touched, "(should be False)")
model.zero_grad()

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


sample y: {"signature": "def solve(nums: list[int]) -> int or None", "plan": "First, count the frequency of each number in the list using a dictionary; then sort the numbers in descending order and return the second-largest unique number, or None if there are fewer than 2 distinct values.", "edge_cases": ["[], [1,1,1]", "[5,4,3,2,1]", "[5,5,5,5,5
h: (96, 3072)  logits: (96, 128256)
LoRA params received gradient: True (should be True)
Frozen base params received gradient: False (should be False)


## O: one OpenAI call, writing code

O's only input is `y` -- it never sees the original `spec`, so x reaches z
entirely through y. O just writes a function body; `z` is the code text that
`Eval_x` runs against hidden tests.

*Troubleshooting:* `TypeError: process() takes no keyword arguments` in an
HTTP-decoding traceback is a stale `brotli` package, not auth --
`pip uninstall brotli`.

In [8]:
from concurrent.futures import ThreadPoolExecutor


def parse_y(y_text):
    """Best-effort parse of G_theta's raw JSON-ish output."""
    match = re.search(r"\{.*\}", y_text, re.DOTALL)
    if not match:
        return {}
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return {}


def extract_code(text):
    """First fenced ```python block, or the raw response if O didn't fence it."""
    match = re.search(r"```(?:python)?\s*(.*?)```", text, re.DOTALL)
    return match.group(1) if match else text


def call_O(y_text):
    """O's only input is y, so x reaches z entirely through y. Returns code (z)."""
    if client is None:
        raise RuntimeError("Set OPENAI_API_KEY in the environment to run this cell.")
    y = parse_y(y_text)
    prompt = (
        f'Implement a Python function with this exact signature: `{y.get("signature")}`.\n'
        f'Approach: {y.get("plan", "no plan given")}\n'
        f'Edge cases to handle: {y.get("edge_cases", [])}\n'
        "Respond with ONLY a fenced Python code block defining that one function."
    )
    resp = client.chat.completions.create(
        model=O_MODEL,
        messages=[
            {"role": "system", "content": (
                "You are a Python coding assistant. You do not see the original "
                "problem statement -- only the guidance below, produced by an "
                "upstream planner.")},
            {"role": "user", "content": prompt},
        ],
    )
    return extract_code(resp.choices[0].message.content or "")


def call_O_batch(y_texts, max_workers=16):
    """Concurrent O-calls, order preserved: they are latency-bound, not compute."""
    if not y_texts:
        return []
    with ThreadPoolExecutor(max_workers=min(max_workers, len(y_texts))) as ex:
        return list(ex.map(call_O, y_texts))

In [9]:
# A few real calls by hand to check the payload/parsing before training.
if client is not None:
    y_text, *_ = rollout(PROBLEMS[0]["spec"])
    code_text = call_O(y_text)
    print("y:", y_text)
    print("code:\n", code_text)
else:
    print("OPENAI_API_KEY not set -- skipping the live O-call smoke test.")

y: {"signature": "def solve(nums: list[int]) -> int or None", "plan": "First, remove duplicates from the list to get unique values. Then, sort the unique values in descending order and return the second element, or None if the list has fewer than 2 distinct values.", "edge_cases": ["", "[]", "[1]", "[1, 1, 1]", "[1, 2]", "[2, 2]", "[2,
code:
 def solve() -> None:
    return None



## Reward and the local unit-test battery

`evalx` is the only signal that costs an O-call. `d_schema`/`d_signature` are
free local checks on G_theta's own output -- the u_m battery. All follow
0=best/1=worst.

*Trust note:* `run_tests` runs O-generated code with `exec` plus a timeout.
Fine for a personal notebook, not a sandbox.

In [10]:
import signal


class _Timeout(Exception):
    pass


def _alarm(signum, frame):
    raise _Timeout()


def run_tests(code_text, tests, fn_name="solve", timeout_s=2):
    """Fraction of hidden tests FAILED (0=all pass, 1=all fail). Unix-only."""
    ns = {}
    try:
        exec(code_text, ns)
    except Exception:
        return 1.0
    fn = ns.get(fn_name)
    if not callable(fn):
        return 1.0

    old_handler = signal.signal(signal.SIGALRM, _alarm)
    failed = 0
    try:
        for args, expected in tests:
            signal.alarm(timeout_s)
            try:
                got = fn(*args)
                if got != expected:
                    failed += 1
            except Exception:
                failed += 1
            finally:
                signal.alarm(0)
    finally:
        signal.signal(signal.SIGALRM, old_handler)
    return failed / len(tests)


def evalx(code_text, problem):
    """0 = O's code passes every hidden test, 1 = fails all."""
    return run_tests(code_text, problem["tests"])


def d_schema(y_text):
    """0 if y is a well-formed {signature, plan, edge_cases} object, else 1."""
    y = parse_y(y_text)
    if not {"signature", "plan"} <= y.keys():
        return 1.0
    # edge_cases must be non-empty: a constant signal is cancelled by its own EMA.
    ec = y.get("edge_cases")
    return 0.0 if isinstance(ec, list) and len(ec) > 0 else 1.0


def d_signature(y_text, problem):
    """0 if G_theta named the function `solve`, else 1. Free, no O-call."""
    # Identifier before the arg list, so "def solve(nums: list) -> int:" scores 0.
    y = parse_y(y_text)
    sig = str(y.get("signature", "")).strip()
    m = re.search(r"([A-Za-z_]\w*)\s*\(", sig)
    return 0.0 if (m and m.group(1) == "solve") else 1.0


# Free (no-O-call) tests for `ours_test`; use_mcp(True) adds d_tools.
EXTRA_TESTS = []


## MCP mode -- O as a frozen tool-using system

In [11]:
import subprocess
import sys

USE_MCP = False                 # flipped by use_mcp()
MAX_TOOL_TURNS = 4              # every turn is billed


def _tool_run_python(code="", timeout_s=2):
    """Run a snippet O wrote, return stdout/stderr.

    Subprocess, not exec + SIGALRM: this executes off the main thread (see
    call_O_batch_mcp), where signal.signal raises, and a runaway thread cannot
    be killed. Not a sandbox -- the child has this interpreter's privileges.
    """
    try:
        proc = subprocess.run([sys.executable, "-c", code], capture_output=True,
                              text=True, timeout=timeout_s)
    except subprocess.TimeoutExpired:
        return f"TimeoutError: still running after {timeout_s}s"
    except Exception as exc:
        return f"{type(exc).__name__}: {exc}"
    return ((proc.stdout or "") + (proc.stderr or "")).strip() or "(ran, no output)"


_DOCS = {
    "sorted": "sorted(iterable, key=None, reverse=False) -> new list; stable.",
    "set": "set(iterable) drops duplicates, unordered. len(set(xs)) = distinct count.",
    "Counter": "collections.Counter(xs).most_common(k) -> [(item, count), ...].",
    "bisect": "bisect.bisect_left(a, x) -> insertion point in a sorted list a.",
    "itertools": "itertools.groupby(sorted(xs)) groups adjacent equal items.",
}


def _tool_lookup_doc(name=""):
    return _DOCS.get(name.strip(),
                     f"No entry for {name!r}. Known: {', '.join(_DOCS)}")


_CHECKLISTS = {
    "list": "empty list; one element; all elements equal; duplicates; negatives",
    "string": "empty string; one char; unbalanced or unclosed; mixed case",
    "number": "zero; negative; the boundary value itself (off-by-one)",
}


def _tool_edge_cases(kind="list"):
    return _CHECKLISTS.get(kind.strip(),
                           "empty input; one element; duplicates; boundary values")


# MCP shape: name, description, JSON input schema. `c` picks the subset and may
# override the description; the schema is fixed here.
MCP_TOOLS = {
    "run_python": dict(
        impl=_tool_run_python,
        description=("Execute a Python snippet and return its stdout/stderr. "
                     "Use it to try your implementation on examples you invent."),
        schema={"type": "object",
                "properties": {"code": {"type": "string",
                                        "description": "Python source to run."}},
                "required": ["code"]}),
    "lookup_doc": dict(
        impl=_tool_lookup_doc,
        description="Look up a one-line reminder for a Python builtin or stdlib helper.",
        schema={"type": "object",
                "properties": {"name": {"type": "string",
                                        "description": "e.g. sorted, Counter, bisect."}},
                "required": ["name"]}),
    "edge_case_checklist": dict(
        impl=_tool_edge_cases,
        description="Get a generic edge-case checklist for an input kind.",
        schema={"type": "object",
                "properties": {"kind": {"type": "string",
                                        "enum": ["list", "string", "number"]}},
                "required": ["kind"]}),
}


def tools_from_c(y):
    """Build the tool list O sees from `c`. Unknown names are dropped, not
    raised -- d_tools punishes that for free. Swap this one function to go
    from the in-process registry to a hosted MCP server."""
    names = y.get("tools") or []
    if isinstance(names, str):
        names = [names]
    notes = y.get("tool_notes") or {}
    if not isinstance(notes, dict):
        notes = {}
    specs = []
    for n in names:
        spec = MCP_TOOLS.get(n)
        if spec is None:
            continue
        desc = notes.get(n) or spec["description"]
        specs.append({"type": "function",
                      "function": {"name": n,
                                   "description": str(desc)[:400],
                                   "parameters": spec["schema"]}})
    return specs


def call_O_mcp(y_text, return_trace=False):
    """O as a frozen compound system: same model plus a host loop that runs the
    tools `c` exposed. O still never sees x. Returns the final code (+ trace)."""
    if client is None:
        raise RuntimeError("Set OPENAI_API_KEY in the environment to run this cell.")
    y = parse_y(y_text)
    tools = tools_from_c(y)
    messages = [
        {"role": "system", "content": (
            "You are a Python coding assistant. You do not see the original "
            "problem statement -- only the guidance below, produced by an "
            "upstream planner, and whatever tools that planner exposed to you. "
            "Use the tools if they help, then answer.")},
        {"role": "user", "content": (
            f'Implement a Python function with this exact signature: `{y.get("signature")}`.\n'
            f'Approach: {y.get("plan", "no plan given")}\n'
            f'Edge cases to handle: {y.get("edge_cases", [])}\n'
            "When you are done, respond with ONLY a fenced Python code block "
            "defining that one function.")},
    ]

    trace = []
    for _ in range(MAX_TOOL_TURNS):
        kwargs = dict(model=O_MODEL, messages=messages)
        if tools:
            kwargs.update(tools=tools, tool_choice="auto")
        msg = client.chat.completions.create(**kwargs).choices[0].message
        calls = msg.tool_calls or []
        if not calls:
            code = extract_code(msg.content or "")
            return (code, trace) if return_trace else code

        # Rebuild the assistant turn by hand: model_dump() carries fields some
        # API versions reject on the way back up.
        messages.append({"role": "assistant", "content": msg.content,
                         "tool_calls": [{"id": tc.id, "type": "function",
                                         "function": {"name": tc.function.name,
                                                      "arguments": tc.function.arguments}}
                                        for tc in calls]})
        for tc in calls:
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            spec = MCP_TOOLS.get(tc.function.name)
            try:
                out = spec["impl"](**args) if spec else f"no such tool: {tc.function.name}"
            except TypeError as exc:                   # args didn't match the schema
                out = f"bad arguments: {exc}"
            out = str(out)[:2000]
            trace.append((tc.function.name, args, out))
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": out})

    # Budget exhausted: force an answer so a tool spiral scores badly, not crashes.
    messages.append({"role": "user",
                     "content": ("Tool budget exhausted. Respond now with ONLY the "
                                 "fenced Python code block.")})
    msg = client.chat.completions.create(model=O_MODEL, messages=messages).choices[0].message
    code = extract_code(msg.content or "")
    return (code, trace) if return_trace else code


def call_O_batch_mcp(y_texts, max_workers=16):
    """Concurrent call_O_mcp, order preserved."""
    if not y_texts:
        return []
    with ThreadPoolExecutor(max_workers=min(max_workers, len(y_texts))) as ex:
        return list(ex.map(call_O_mcp, y_texts))


def build_prompt_mcp(problem_spec):
    """build_prompt, extended to ask for the c half of y."""
    catalog = ", ".join(f'"{n}"' for n in MCP_TOOLS)
    system = (
        "You are a coding coach who also configures the engineer's workspace. "
        "The engineer who implements this will NOT see the problem statement -- "
        "they will see only your guidance and the tools you expose to them. "
        "Restate the exact function signature, give a short approach, list the "
        "edge cases that matter, then choose which tools to expose (available: "
        f"{catalog}) and write the description the engineer will read for each. "
        "Respond with exactly one line of JSON: "
        '{"signature": "<fn_name(args)>", "plan": "<1-2 sentence approach>", '
        '"edge_cases": ["<case>", ...], "tools": ["<name>", ...], '
        '"tool_notes": {"<name>": "<what it is for>"}}.'
    )
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": problem_spec}]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def d_tools(y_text):
    """0 if c names only real tools and describes every one it names, else 1.
    Free, no O-call: separates a broken interface config from a broken plan."""
    y = parse_y(y_text)
    names = y.get("tools")
    if not isinstance(names, list) or not names:
        return 1.0
    if any(n not in MCP_TOOLS for n in names):
        return 1.0
    notes = y.get("tool_notes")
    if not isinstance(notes, dict) or any(n not in notes for n in names):
        return 1.0
    return 0.0


_build_prompt_plain, _call_O_batch_plain = build_prompt, call_O_batch


MCP_T_MAX = 224
_T_MAX_PLAIN = None


def use_mcp(on=True, t_max=None):
    """Turn MCP mode on/off for everything downstream: rebinding build_prompt,
    call_O_batch and EXTRA_TESTS is the whole integration (they are looked up
    as globals at call time). Estimators and Eval_x are untouched.

    Raising T_MAX is NOT optional: at 96, every y truncates mid-edge_cases,
    parse_y returns {} and the run silently scores f=1.0 on everything.
    """
    global USE_MCP, build_prompt, call_O_batch, EXTRA_TESTS, T_MAX, _T_MAX_PLAIN
    USE_MCP = bool(on)
    build_prompt = build_prompt_mcp if USE_MCP else _build_prompt_plain
    call_O_batch = call_O_batch_mcp if USE_MCP else _call_O_batch_plain
    EXTRA_TESTS = [("u_tools", d_tools)] if USE_MCP else []

    if t_max is not False:
        if USE_MCP:
            if _T_MAX_PLAIN is None:
                _T_MAX_PLAIN = T_MAX
            T_MAX = int(t_max or MCP_T_MAX)
        elif _T_MAX_PLAIN is not None:
            T_MAX, _T_MAX_PLAIN = _T_MAX_PLAIN, None

    print(f"MCP mode {'ON' if USE_MCP else 'off'}: y = "
          f"{'(p, c)' if USE_MCP else 'p only'}, T_MAX = {T_MAX}")


In [12]:
# Smoke test: a few real calls to see whether c does anything. Leaves MCP OFF.
if client is not None:
    use_mcp(True)
    problem = PROBLEMS[0]
    y_text, *_ = rollout(problem["spec"])
    y = parse_y(y_text)
    code_text, trace = call_O_mcp(y_text, return_trace=True)

    print("y (p, c):", json.dumps(y, indent=2)[:800])
    print("\nc -> tools O actually got:",
          [t["function"]["name"] for t in tools_from_c(y)] or "(none)")
    print("d_tools:", d_tools(y_text), " d_schema:", d_schema(y_text))
    print("\ntrajectory:")
    for name, args, out in trace:
        print(f"  {name}({args}) -> {out[:120]!r}")
    print("\nz (final code):\n", code_text)
    print("f = Eval_x(z):", evalx(code_text, problem))

    use_mcp(False)
else:
    print("OPENAI_API_KEY not set -- skipping the MCP smoke test.")

MCP mode ON: y = (p, c), T_MAX = 224
y (p, c): {
  "signature": "def solve(nums): return sorted(set(nums))[1] if len(set(nums)) > 1 else None",
  "plan": "Sort the unique values in the list, then return the second element if it exists; otherwise, return None.",
  "edge_cases": [
    "nums = [5, 5, 9]",
    "nums = [1, 2, 3]",
    "nums = [1]",
    "nums = []"
  ],
  "tools": [
    "run_python",
    "lookup_doc"
  ],
  "tool_notes": {
    "run_python": "Run the provided Python code to execute the function and test edge cases",
    "lookup_doc": "Look up the documentation for set() and sorted() functions to understand their behavior in the context of this problem"
  }
}

c -> tools O actually got: ['run_python', 'lookup_doc']
d_tools: 0.0  d_schema: 0.0

trajectory:

z (final code):
 def solve(nums): return sorted(set(nums))[1] if len(set(nums)) > 1 else None

f = Eval_x(z): 0.0
MCP mode off: y = p only, T_MAX = 96


In [13]:
# MCP mode for everything below: y = (p, c), O gets tools, T_MAX -> 224.
# Must come AFTER the smoke test. use_mcp(False) goes back to plain O.
# Costs ~+158% input tokens and ~1.5x wall clock per sample.
use_mcp(True)

MCP mode ON: y = (p, c), T_MAX = 224


## Our algorithm: Q control variates, and Delta variance descent

`reinforce_b` and `ours_test` only reweight log p_theta; neither is derived to
minimize anything. These two are:

- **`qcv`** -- per-token Q control variate with an exact vocab-marginalized
  correction:
  $$\hat g=\sum_t\Big[\big(f-\hat Q_t(y_{<t},y_t)\big)\nabla_\theta\log p_t(y_t)
  \;+\;\nabla_\theta\!\!\sum_{v\in\mathcal V}\! p_\theta(v\mid y_{<t})\,\hat Q_t(y_{<t},v)\Big]$$
  The correction cancels the control variate in expectation at every position,
  so this is unbiased for **any** `Qhat` -- its quality moves only variance.

- **`qcv_delta`** -- the variance-*minimizing* one. Adds a residual head,
  c = Qhat + Delta, and trains Delta by descending E||ghat||^2. Since E[ghat]
  does not depend on Delta (Lemma 1), that is exactly minimizing Var(ghat).
  Costs no extra O-calls: the sample already drawn is reused via double
  backprop, with no reparameterization.

Scale is what changes from the toy: V=128256, so the correction is a real
(T,V) tensor per step and `Qhat` cannot be a dense (d_model, V) head. If
`qcv_delta` OOMs, lower T_MAX.

In [14]:
import torch.nn as nn
from collections import deque


class QHead(nn.Module):
    """Qhat_t(y_{<t}, v) = <A h_t, B e_v>, tied to the frozen input embeddings.

    A dense (d_model, V) head would be ~394M params here -- 170x the LoRA
    policy it is meant to help, and unfittable from a few dozen O-calls.
    """

    def __init__(self, d_model, emb, r=32, zero_init=False):
        super().__init__()
        self.A = nn.Linear(d_model, r, bias=False)
        self.B = nn.Linear(d_model, r, bias=False)
        self._emb = (emb,)   # tuple-wrapped: keep the ~788MB frozen matrix unregistered
        nn.init.normal_(self.A.weight, std=0.02)
        # zero_init outputs exactly 0, so c = Qhat + Delta starts as plain Qhat.
        nn.init.zeros_(self.B.weight) if zero_init else \
            nn.init.normal_(self.B.weight, std=0.02)

    def _tok(self, ids=None):
        """B e_v for every v (ids=None) or just the taken tokens."""
        E = self._emb[0]
        E = E if ids is None else E[ids]
        return tF.linear(E, self.B.weight.to(E.dtype)).float()

    def forward(self, h):
        """Full (T, V) table -- needed only for the marginalized correction."""
        return self.A(h.float()) @ self._tok().T

    def taken(self, h, y_ids):
        """Just Qhat_t(y_{<t}, y_t), without materializing the (T, V) table."""
        return (self.A(h.float()) * self._tok(y_ids)).sum(-1)


def fit_q(qhead, qopt, replay, rng, n_steps=8, bs=4):
    """Return-regression on replay: min sum_t (Qhat_t(y_{<t}, y_t) - f)^2."""
    for _ in range(n_steps):
        idx = rng.integers(0, len(replay), size=min(bs, len(replay)))
        loss = 0.0
        for i in idx:
            h_i, y_i, f_i = replay[i]
            q = qhead.taken(h_i.to(device), y_i.to(device))
            loss = loss + ((q - f_i) ** 2).mean()
        qopt.zero_grad()
        (loss / len(idx)).backward()
        qopt.step()

## Four estimators, one small loop

All four share the rollout, the O-call and the reward, differing only in what
multiplies log p_theta (plus the correction term for the Q-based pair).

- `reinforce_b` -- scalar EMA baseline, per problem.
- `ours_test` -- `a_t^test` from `evalx` plus the free `d_schema`/`d_signature`.
- `qcv` -- learned-Q control variate + vocab-marginalized correction.
- `qcv_delta` -- `qcv` plus the `Delta` head trained by variance descent.

In [15]:
class EMABaseline:
    def __init__(self, decay=0.9):
        self.decay = decay
        self.value = {}

    def get(self, key):
        return self.value.get(key)

    def update(self, key, d):
        old = self.value.get(key)
        self.value[key] = d if old is None else self.decay * old + (1 - self.decay) * d


def a_test(T, fired, baselines):
    """fired: list of (test_name, d). Every support is the full span here, so
    a_t is the same value at every position."""
    coeff = 0.0
    for name, d in fired:
        b = baselines.get(name)
        if b is None:
            b = d
        coeff += (d - b)
        baselines.update(name, d)
    return torch.full((T,), coeff, dtype=torch.float32)


def _surrogate(method, item, f, pid, problem, baseline, test_baselines,
               qhead, dhead):
    """Per-sample surrogate whose grad w.r.t. theta is that sample's ghat."""
    lp, y_ids, h, logits = item["lp"], item["y_ids"], item["h"], item["logits"]
    T = y_ids.shape[0]

    # Baselines keyed PER PROBLEM: these problems differ enough in difficulty
    # that one global baseline would mostly inject instance-difficulty noise.
    if method == "reinforce_b":
        b = baseline.get(pid)
        if b is None:
            b = f
        baseline.update(pid, f)
        return (f - b) * lp.sum()

    if method == "ours_test":
        fired = [(f"u_global@{pid}", f),
                 (f"u_schema@{pid}", d_schema(item["y_text"])),
                 (f"u_signature@{pid}", d_signature(item["y_text"], problem))]
        fired += [(f"{n}@{pid}", fn(item["y_text"])) for n, fn in EXTRA_TESTS]
        a = a_test(T, fired, test_baselines).to(lp.device)
        return (a * lp).sum()

    with torch.no_grad():
        Q = qhead(h)                               # (T, V), fixed w.r.t. theta
    if method == "qcv_delta":
        Q = Q + dhead(h)                           # grad flows into Delta only
    ar = torch.arange(T, device=y_ids.device)
    # score term + exact vocab-marginalized correction
    return ((f - Q[ar, y_ids]) * lp).sum() + (torch.softmax(logits, -1) * Q).sum()


def train(method, n_calls, batch=4, lr=1e-4, seed=0, q_lr=1e-3, delta_lr=1e-3,
          q_steps=8, replay_cap=64, return_heads=False, verbose=True):
    """Train G_theta with one estimator on a budget of `n_calls` O-calls.

    The budget is counted in O-calls, the resource the comparison is matched
    on; `batch` samples are drawn per update, so keep it identical across
    methods. Memory caps it: the Q-based methods hold an (T, V) table per row.
    """
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    theta = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.Adam(theta, lr=lr)
    baseline, test_baselines = EMABaseline(), EMABaseline()
    history = []
    qhead = dhead = None

    needs_q = method in ("qcv", "qcv_delta")
    if needs_q:
        emb = base.get_input_embeddings().weight
        qhead = QHead(base.config.hidden_size, emb).to(device)
        qopt = torch.optim.Adam(qhead.parameters(), lr=q_lr)
        replay = deque(maxlen=replay_cap)
        if method == "qcv_delta":
            dhead = QHead(base.config.hidden_size, emb, zero_init=True).to(device)
            dopt = torch.optim.Adam(dhead.parameters(), lr=delta_lr)

    n_updates, spent = max(1, n_calls // batch), 0
    for step in range(n_updates):
        problems = [PROBLEMS[(spent + i) % len(PROBLEMS)] for i in range(batch)]
        items, _ = rollout_batch([p["spec"] for p in problems],
                                 double_backward=(method == "qcv_delta"),
                                 need_h=needs_q)
        codes = call_O_batch([it["y_text"] for it in items])
        fs = [evalx(c, p) for c, p in zip(codes, problems)]
        spent += batch

        surrs = [_surrogate(method, it, f, p["id"], p, baseline, test_baselines,
                            qhead, dhead)
                 for it, f, p in zip(items, fs, problems)]

        if method == "qcv_delta":
            # Variance descent on Delta. Objective is mean_i ||ghat_i||^2, per
            # sample -- ||mean_i ghat_i||^2 would train Delta to cancel the true
            # gradient. Backward each immediately: one grad-of-grad graph alive.
            dopt.zero_grad()
            g_acc = [torch.zeros_like(p) for p in theta]
            for k, surr in enumerate(surrs):
                keep = k < len(surrs) - 1        # shared forward graph
                g_vec = torch.autograd.grad(surr, theta, create_graph=True,
                                            retain_graph=True, allow_unused=True)
                g_vec = [g if g is not None else torch.zeros_like(p)
                         for g, p in zip(g_vec, theta)]
                for acc, g in zip(g_acc, g_vec):
                    acc += g.detach() / batch
                (sum((g ** 2).sum() for g in g_vec) / batch).backward(
                    retain_graph=keep)
                del g_vec
            dopt.step()

            opt.zero_grad()                      # reuse g_acc: no extra backward
            for prm, g in zip(theta, g_acc):
                prm.grad = g
            opt.step()
        else:
            opt.zero_grad()
            (sum(surrs) / batch).backward()
            opt.step()

        if needs_q:
            for it, f in zip(items, fs):
                replay.append((it["h"].detach().cpu(), it["y_ids"].cpu(), f))
            fit_q(qhead, qopt, replay, rng, n_steps=q_steps)

        del items, surrs
        history.extend(fs)
        if verbose:
            print(f"[{method}] update {step:3d}  calls={spent:4d}  "
                  f"f_mean={np.mean(fs):.2f}")
    return (history, qhead, dhead) if return_heads else history

## Run + compare at a matched budget

`N_CALLS` O-calls per method: a sanity check that the loop works, not a
rigorous comparison (single seed, no multi-seed statistics -- see
`toy_token_q/README.md`'s E3).

That budget is nowhere near enough to fit `Qhat` well, so expect the Q-based
pair's variance advantage to be muted. Both stay unbiased regardless -- that
is what the correction term buys. Raise `N_CALLS` to see it pay off.

In [2]:
# Budget is in O-CALLS; BATCH only sets how many are drawn per update.
# BATCH=4 is the largest all four methods share (qcv_delta peaks at 33.5GB and
# OOMs at 8). Keep it IDENTICAL across methods -- drop all four together if needed.
N_CALLS, BATCH = 120, 4

# reset_lora()
hist_baseline = train("reinforce_b", n_calls=N_CALLS, batch=BATCH, seed=0)

# reset_lora()
# hist_ours = train("ours_test", n_calls=N_CALLS, batch=BATCH, seed=0)

reset_lora()
hist_qcv = train("qcv", n_calls=N_CALLS, batch=BATCH, seed=0)

NameError: name 'train' is not defined

In [20]:
# Persist the runs so a kernel restart never re-spends their O-calls. Keyed by
# mode: MCP and plain histories are not comparable. RESULTS must stay UNSTAMPED
# (it is the read path too); the date goes on a snapshot copy instead.
from datetime import datetime

RESULTS = f"results_{'mcp' if USE_MCP else 'plain'}.json"


def _snapshot_path():
    return f"{RESULTS[:-5]}_{datetime.now():%m-%d-%y_%H%M%S}.json"


def save_hist(**named):
    d = json.load(open(RESULTS)) if os.path.exists(RESULTS) else {}
    d.update({k: [float(x) for x in v] for k, v in named.items()})
    json.dump(d, open(RESULTS, "w"), indent=1)
    snap = _snapshot_path()
    json.dump(d, open(snap, "w"), indent=1)
    print(f"saved {list(named)} -> {RESULTS} (+ snapshot {snap}); "
          f"file now holds {sorted(d)}")


def load_hist(*names):
    d = json.load(open(RESULTS)) if os.path.exists(RESULTS) else {}
    missing = [n for n in names if n not in d]
    if missing:
        raise KeyError(f"{missing} not in {RESULTS} (has {sorted(d)}) -- "
                       "rerun the training cell above.")
    return [d[n] for n in names]


# on-disk key -> notebook variable
_HISTS = [("reinforce_b", "hist_baseline"), ("ours_test", "hist_ours"),
          ("qcv", "hist_qcv"), ("qcv_delta", "hist_qcv_delta")]


def sync_hist():
    """Two-way sync between kernel memory and RESULTS, one history at a time.

    Saves what is in memory, restores what is not. Per-history on purpose: a
    single all-at-once save raises NameError and saves NOTHING when only some
    of the four have been trained, losing O-calls already spent.
    """
    g = globals()
    on_disk = json.load(open(RESULTS)) if os.path.exists(RESULTS) else {}
    to_save = {k: g[v] for k, v in _HISTS if v in g}
    if to_save:
        save_hist(**to_save)
        on_disk.update(to_save)
    restored = [v for k, v in _HISTS if v not in g and k in on_disk]
    for k, v in _HISTS:
        if v not in g and k in on_disk:
            g[v] = on_disk[k]
    missing = [v for k, v in _HISTS if v not in g]
    print("in memory:", sorted(to_save) or "none",
          "| restored:", restored or "none",
          "| still missing:", missing or "none")


sync_hist()

saved ['qcv'] -> results_mcp_08-28-26_185116.json; file now holds ['qcv']
in memory: ['qcv'] | restored: none | still missing: ['hist_baseline', 'hist_ours', 'hist_qcv_delta']


In [21]:
# Own cell so a rerun does not re-spend the other three methods' O-calls.
# return_heads=True hands back the trained Qhat/Delta for the variance probe;
# they are live modules and are NOT checkpointed, unlike the histories.
gc.collect()
torch.cuda.empty_cache()
print(f"free before qcv_delta: "
      f"{torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB (needs ~33.5 at BATCH=4)")

reset_lora()
hist_qcv_delta, qhead_trained, dhead_trained = train(
    "qcv_delta", n_calls=N_CALLS, batch=BATCH, seed=0, return_heads=True)

sync_hist()

free before qcv_delta: 70.4 GiB (needs ~33.5 at BATCH=4)
[qcv_delta] update   0  calls=   4  f_mean=0.56
[qcv_delta] update   1  calls=   8  f_mean=0.25
[qcv_delta] update   2  calls=  12  f_mean=0.12
[qcv_delta] update   3  calls=  16  f_mean=0.05
[qcv_delta] update   4  calls=  20  f_mean=0.25
[qcv_delta] update   5  calls=  24  f_mean=0.50
[qcv_delta] update   6  calls=  28  f_mean=0.00
[qcv_delta] update   7  calls=  32  f_mean=0.00
[qcv_delta] update   8  calls=  36  f_mean=0.25
[qcv_delta] update   9  calls=  40  f_mean=0.20
[qcv_delta] update  10  calls=  44  f_mean=0.00
[qcv_delta] update  11  calls=  48  f_mean=0.00
[qcv_delta] update  12  calls=  52  f_mean=0.35
[qcv_delta] update  13  calls=  56  f_mean=0.25
[qcv_delta] update  14  calls=  60  f_mean=0.00
[qcv_delta] update  15  calls=  64  f_mean=0.30
[qcv_delta] update  16  calls=  68  f_mean=0.00
[qcv_delta] update  17  calls=  72  f_mean=0.00
[qcv_delta] update  18  calls=  76  f_mean=0.44
[qcv_delta] update  19  calls= 

In [1]:
def rolling(x, w=5):
    x = np.asarray(x, dtype=float)
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w) / w, mode="valid")

curves = [("reinforce_b", hist_baseline), ("ours_test (a_t^test)", hist_ours),
          ("qcv", hist_qcv), ("qcv_delta (variance descent)", hist_qcv_delta)]

plt.figure(figsize=(7, 4))
for label, hist in curves:
    plt.plot(rolling(hist), label=label)
plt.xlabel("O-call")
plt.ylabel("regret f (rolling mean, 0 = best)")
plt.title("Four estimators at a matched O-call budget")
plt.legend()
plt.tight_layout()
plt.show()

for label, hist in curves:
    h = np.asarray(hist, dtype=float)
    print(f"{label:32s} mean={h.mean():.3f}  first10={h[:10].mean():.3f}  "
          f"last10={h[-10:].mean():.3f}")

NameError: name 'hist_baseline' is not defined

## Measuring the thing the theory actually claims

The regret curves are a noisy proxy: the theory predicts a gap in
**Var(ghat)**, not regret after 30 updates, and each run walks its own
trajectory, so comparing them confounds variance with sampling luck.

`variance_probe` measures it directly: freeze theta, draw one batch of M
samples, score every estimator on those same samples. Paired, so much sharper
-- and M O-calls total, not M per estimator, since pass 2 re-runs only the
free teacher-forced forward.

`fit_frac` splits the draw: baselines, `Qhat` and `Delta` are fit on the first
half, `Var(ghat)` measured on the second. Not optional for `Delta`, which
otherwise wins by memorising the samples it is scored on.

Read two columns. `||E[ghat]||` should **agree** (all unbiased); `Var(ghat)`
is where they may differ and is the number the method is about. If `qcv` is
not below `reinforce_b`, `Qhat` is undertrained.

In [20]:
class GradStats:
    """Streaming mean and second moment of gradient estimates.

    var_total() is E||g||^2 - ||E[g]||^2 (trace of the covariance); se() is how
    far the sample mean is expected to sit from the true E[ghat].
    """

    def __init__(self):
        self.sum_g, self.sum_sq, self.n = None, 0.0, 0
        self.sq = []                       # per-sample ||g||^2, for bootstrap

    def add(self, g):
        self.sum_g = g.clone() if self.sum_g is None else self.sum_g + g
        s = float((g ** 2).sum())
        self.sum_sq += s
        self.sq.append(s)
        self.n += 1

    def mean(self):
        return self.sum_g / self.n

    def var_total(self):
        return self.sum_sq / self.n - float((self.mean() ** 2).sum())

    def se(self):
        return (max(self.var_total(), 0.0) / self.n) ** 0.5


def _flat_grad(surr, theta, create_graph=False):
    gs = torch.autograd.grad(surr, theta, retain_graph=True,
                             create_graph=create_graph, allow_unused=True)
    return [g if g is not None else torch.zeros_like(p)
            for g, p in zip(gs, theta)]


def _flat_cat(gs):
    return torch.cat([g.reshape(-1) for g in gs])


def _qcv_surrogate(f, lp, y_ids, h, logits, qhead, dhead, delta_grad=True):
    """One sample's ghat surrogate under c = Qhat (+ Delta).

    Qhat is always detached -- it is a control variate, not something theta or
    Delta optimizes through. Delta keeps its graph only while being FITTED.
    """
    ar = torch.arange(y_ids.shape[0], device=y_ids.device)
    with torch.no_grad():
        Q = qhead(h)
    if dhead is not None:
        if delta_grad:
            Q = Q + dhead(h)
        else:
            with torch.no_grad():
                Q = Q + dhead(h)
    return ((f - Q[ar, y_ids]) * lp).sum() + (torch.softmax(logits, -1) * Q).sum()


def fit_delta(dhead, dopt, groups, qhead, theta, n_steps=200, verbose=True):
    """Self-fit Delta by variance descent at the CURRENT theta (fit_q's counterpart).

    Objective is mean_i ||ghat_i||^2, per sample; since E[ghat] does not depend
    on Delta (Lemma 1), descending it is exactly descending Var(ghat). Delta is
    zero-initialised, so step 0 is identically qcv -- any failure to beat qcv
    from here is optimization or generalization, never capacity.
    """
    hist = []
    for step in range(n_steps):
        gr = groups[step % len(groups)]
        # Whole groups only: rollout_batch derives the span from the padded
        # prompt length, so re-scoring a slice would misalign the tokens.
        items, _ = rollout_batch(gr["specs"], double_backward=True,
                                 gens=gr["gens"])
        n = len(items)
        dopt.zero_grad()
        tot = 0.0
        for i, (it, r) in enumerate(zip(items, gr["recs"])):
            surr = _qcv_surrogate(r["f"], it["lp"], it["y_ids"], it["h"],
                                  it["logits"], qhead, dhead)
            g_vec = _flat_grad(surr, theta, create_graph=True)
            obj = sum((g ** 2).sum() for g in g_vec) / n
            # One shared forward graph per group: free it on the last backward.
            obj.backward(retain_graph=(i < n - 1))
            tot += float(obj.detach())
            del g_vec, obj, surr
        dopt.step()
        hist.append(tot)
        if verbose and (step % max(1, n_steps // 5) == 0 or step == n_steps - 1):
            # Per-GROUP objective: consecutive steps visit different groups, so
            # compare steps that are len(groups) apart.
            print(f"    fit_delta step {step:4d}  [group {step % len(groups)}]"
                  f"  mean||ghat||^2 = {tot:.4g}")
        del items
    model.zero_grad(set_to_none=True)   # the double backward also hits theta
    return hist


def variance_probe(M=64, batch=4, qhead=None, dhead=None, seed=123,
                   q_fit_steps=200, delta_steps=200, delta_lr=1e-3,
                   fit_frac=0.5, n_boot=4000, verbose=True):
    """Measure Var(ghat) for every estimator on the SAME samples, at frozen theta.

    Paired on purpose: in the training runs each estimator walks its own
    trajectory, so comparing them confounds variance with sampling luck. Costs
    M O-calls TOTAL -- pass 2 only re-runs the free teacher-forced forward.

    fit_frac splits the draw: everything learned (baselines, Qhat, Delta) is fit
    on the first fraction and Var(ghat) measured on the rest. Not optional for
    Delta, which otherwise wins by memorising the samples it is scored on.
    """
    theta = [p for p in model.parameters() if p.requires_grad]
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)

    # ---- pass 1: spend the O-call budget, once, in batches ----------------
    groups, drawn = [], 0
    while drawn < M:
        bs = min(batch, M - drawn)
        problems = [PROBLEMS[(drawn + i) % len(PROBLEMS)] for i in range(bs)]
        items, gens = rollout_batch([p["spec"] for p in problems])
        codes = call_O_batch([it["y_text"] for it in items])
        recs = []
        for it, c, p in zip(items, codes, problems):
            # Same battery `ours_test` is TRAINED with, EXTRA_TESTS included.
            tests = {"u_schema": d_schema(it["y_text"]),
                     "u_signature": d_signature(it["y_text"], p)}
            tests.update({n: fn(it["y_text"]) for n, fn in EXTRA_TESTS})
            recs.append(dict(problem=p, pid=p["id"], f=evalx(c, p),
                             h=it["h"].detach().cpu(), y_ids=it["y_ids"].cpu(),
                             tests=tests))
        groups.append(dict(specs=[p["spec"] for p in problems],
                           gens=gens.detach(), recs=recs))
        drawn += bs
        del items

    if fit_frac and fit_frac > 0:
        k = min(max(1, int(round(len(groups) * fit_frac))), len(groups) - 1)
        fit_groups, eval_groups = groups[:k], groups[k:]
    else:
        fit_groups = eval_groups = groups          # legacy: fit == eval
    fit_flat = [r for gr in fit_groups for r in gr["recs"]]
    eval_flat = [r for gr in eval_groups for r in gr["recs"]]

    # ---- baselines: learned on the fit split, like Qhat and Delta ---------
    test_names = list(fit_flat[0]["tests"].keys())

    def baselines_for(get):
        by, allv = {}, [get(r) for r in fit_flat]
        for r in fit_flat:
            by.setdefault(r["pid"], []).append(get(r))
        return ({k: float(np.mean(v)) for k, v in by.items()},
                float(np.mean(allv)))

    b_f, b_f_all = baselines_for(lambda r: r["f"])
    b_t = {n: baselines_for(lambda r, n=n: r["tests"][n]) for n in test_names}

    methods = ["reinforce_b", "ours_test", "qcv"]
    emb = base.get_input_embeddings().weight
    if qhead is None:
        qhead = QHead(base.config.hidden_size, emb).to(device)
        fit_q(qhead, torch.optim.Adam(qhead.parameters(), lr=1e-3),
              [(r["h"], r["y_ids"], r["f"]) for r in fit_flat],
              rng, n_steps=q_fit_steps)
        q_src = f"self-fit, {q_fit_steps} regression steps on fit split"
    else:
        q_src = "imported (fit at another theta)"

    if dhead is None and delta_steps:
        dhead = QHead(base.config.hidden_size, emb, zero_init=True).to(device)
        if verbose:
            print(f"  fitting Delta by variance descent ({delta_steps} steps)...")
        fit_delta(dhead, torch.optim.Adam(dhead.parameters(), lr=delta_lr),
                  fit_groups, qhead, theta, n_steps=delta_steps, verbose=verbose)
        d_src = f"self-fit, {delta_steps} variance-descent steps on fit split"
    elif dhead is not None:
        d_src = "imported (fit at another theta)"
    else:
        d_src = None
    if dhead is not None:
        methods.append("qcv_delta")

    # ---- pass 2: re-score the EVAL split under every estimator -------------
    stats = {m: GradStats() for m in methods}
    for gr in eval_groups:
        items, _ = rollout_batch(gr["specs"], gens=gr["gens"])
        for it, r in zip(items, gr["recs"]):
            lp, y_ids, h, logits = it["lp"], it["y_ids"], it["h"], it["logits"]
            pid, f = r["pid"], r["f"]
            for m in methods:
                if m == "reinforce_b":
                    surr = (f - b_f.get(pid, b_f_all)) * lp.sum()
                elif m == "ours_test":
                    c = f - b_f.get(pid, b_f_all)
                    for n in test_names:
                        by, allv = b_t[n]
                        c = c + (r["tests"][n] - by.get(pid, allv))
                    surr = c * lp.sum()
                else:
                    surr = _qcv_surrogate(f, lp, y_ids, h, logits, qhead,
                                          dhead if m == "qcv_delta" else None,
                                          delta_grad=False)
                stats[m].add(_flat_cat(_flat_grad(surr, theta)))
        del items

    # ---- report -----------------------------------------------------------
    ref = methods[0]
    sq = {m: np.asarray(stats[m].sq) for m in methods}
    idx = rng.integers(0, len(sq[ref]), size=(n_boot, len(sq[ref])))
    ref_var = stats[ref].var_total()

    def _ratio_ci(m):
        """Paired bootstrap CI for E||ghat_m||^2 / E||ghat_ref||^2: same
        resampled indices for both. Guarded -- the reference can be zero."""
        den, num = sq[ref][idx].mean(1), sq[m][idx].mean(1)
        ok = den > 0
        if not ok.any():
            return float("nan"), float("nan")
        return tuple(np.percentile(num[ok] / den[ok], [2.5, 97.5]))

    if verbose:
        print(f"\npaired variance probe: theta frozen, {M} O-calls total")
        print(f"  draw {M} = fit {len(fit_flat)} + eval {len(eval_flat)}"
              f"{'  (NO SPLIT -- leaky)' if fit_groups is eval_groups else ''}")
        print(f"  Qhat : {q_src}")
        print(f"  Delta: {d_src or 'absent (qcv_delta not measured)'}")
        print(f"  ours_test battery: {', '.join(['f'] + test_names)}\n")
        print(f"  {'estimator':14s} {'Var(ghat)':>11s} {'ratio':>8s} "
              f"{'95% CI (paired)':>18s} {'||E[ghat]||':>12s} {'+/- SE':>9s}")
        for m in methods:
            v, st = stats[m].var_total(), stats[m]
            lo, hi = _ratio_ci(m)
            rat = f"{v / ref_var:7.3f}x" if ref_var > 0 else "     n/a"
            ci = "[%.2f, %.2f]" % (lo, hi) if lo == lo else "n/a"
            print(f"  {m:14s} {v:11.4g} {rat:>8s} {ci:>18s} "
                  f"{float(st.mean().norm()):12.4g} {st.se():9.3g}")
        if ref_var <= 0:
            print(f"\n  !! {ref}'s Var(ghat) is 0 -- its gradient vanished on every")
            print("     eval sample, which happens when each problem's f equals its")
            print("     baseline. Ratios are undefined; raise M or vary the problems.")

        worst = max(stats[m].se() / max(float(stats[m].mean().norm()), 1e-12)
                    for m in methods)
        print("\nVar(ghat) is the number the method is about -- lower is better.")
        print("||E[ghat]|| should agree across estimators (all unbiased), but")
        print("only up to +/- SE:")
        if worst > 0.5:
            print(f"  !! SE is {worst:.1f}x the mean, so ||E[ghat]|| is NOT resolved.")
            print("     Raise M before reading anything into that column.")
        else:
            print(f"  means are resolved (worst SE/||E|| = {worst:.2f}); compare them.")
        if "qcv_delta" in methods and d_src and d_src.startswith("self-fit"):
            print("\nDelta starts at exactly zero, so qcv_delta starts as qcv. A ratio")
            print("above qcv's therefore means the variance descent failed to")
            print("generalize off its fit split -- not that the method lacks capacity.")
    return stats


In [21]:
# Probe 1 -- at the TRAINED theta, with the heads trained there. MUST come
# before reset_lora(): a Qhat/Delta fit at one theta is a poor control variate
# at another, and that mismatch alone moved qcv from 0.292x to 3.563x.
M_PROBE = 64

probe_trained = variance_probe(M=M_PROBE, batch=4,
                               qhead=qhead_trained, dhead=dhead_trained)

# Probe 2 -- fresh theta, Qhat AND Delta both self-fit here and scored on the
# held-out half: the apples-to-apples test. reset_lora() discards the trained
# policy (LoRA weights are not checkpointed; the f histories are safe in RESULTS).
reset_lora()
probe_fresh = variance_probe(M=M_PROBE, batch=4)


paired variance probe: M=32 samples, 32 O-calls total, theta frozen

  estimator         Var(ghat)     ratio  ||E[ghat]||     +/- SE
  reinforce_b            1560    1.000x        6.786       6.98
  ours_test         1.456e+04    9.332x        20.92       21.3
  qcv                   455.2    0.292x        4.252       3.77

Var(ghat) is the number the method is about -- lower is better, and
the ratio column is the paired comparison. ||E[ghat]|| should agree
across estimators (all are unbiased), but only up to +/- SE:
  !! SE is 1.0x the mean here, so ||E[ghat]|| is NOT resolved.
     Raise M before reading anything into that column.

paired variance probe: M=32 samples, 32 O-calls total, theta frozen

  estimator         Var(ghat)     ratio  ||E[ghat]||     +/- SE
  reinforce_b            1829    1.000x        7.457       7.56
  ours_test         1.455e+04    7.957x         21.2       21.3
  qcv                    6515    3.563x        14.57       14.3
  qcv_delta              8667   

## Takeaway

The whole loop end to end: a local LoRA-tuned G_theta proposes a coaching
note, a frozen OpenAI model writes code from it alone, and a score-function
estimator pushes G_theta toward notes that make O pass more hidden tests --
without ever differentiating through O.

Left out, and where to find it:
- Other estimators (`rloo`, `pathwise`, the `qcv_exact` theory floor) and the
  exact-Q diagnostics -- `toy_token_q/{token_q_toy,dfl_toy}.py`.
- Multi-seed, calls-to-threshold comparisons -- `toy_token_q/README.md` E1-E4.
- The fuller framework and open questions -- `black-box DFL/mcp_prototype_plan.md`.